# Stage 4: Multi-Paradigm Mining & Model Building
## Food Adulteration Risk Prediction Using Data Mining Techniques
**Course:** Data Mining Techniques (DMT), B.Tech Information Technology  
**Institution:** K. Ramakrishnan College of Technology (Autonomous), Trichy  
**Team:** Jayanth S (2403811720521023), Jithin AK (2403811720521025), Keerthivasan RP (2403811720521026)

---
> ⚠️ **CRITICAL DISCLAIMER: SYNTHETIC DATA — GENERATED FOR ACADEMIC DEMONSTRATION**  
> *All model training and evaluations are conducted on domain-modeled synthetic data grounded in FSSAI regulations. Not for real regulatory decision-making.*
---


### 1. Architectural Scope & Multi-Paradigm Data Mining
This stage implements the core computational mining engines defined in Stage 4 of our system architecture:
1. **Supervised Classification & Hyperparameter Tuning**:
   - Primary Architectural Models: **Decision Tree**, **Random Forest**, **Gaussian Naive Bayes**, **Support Vector Machine (SVM)**
   - Algorithmic Extensions: **Logistic Regression**, **K-Nearest Neighbors (KNN)**, **Gradient Boosting**
   - 5-Fold Stratified Cross-Validation (`GridSearchCV`) optimizing for **Recall (Sensitivity)** on the adulterated class.
2. **Unsupervised Cluster Discovery**:
   - **K-Means Clustering** evaluated across $k \in [2, 6]$ via the **Elbow Method (Inertia)** and **Silhouette Coefficient Analysis** to investigate whether contaminated samples form natural, unguided geometric clusters in feature space.
3. **Association Rule Mining**:
   - **Apriori Algorithm** applied to domain-discretized transactional items to uncover high-lift co-occurrence rules (e.g. `{Non-Permitted Preservative, Expired Stock} \implies {Adulterated}`).


In [ ]:
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Image, display

# Add project root to path
sys.path.append(os.path.abspath('..'))
from src.models import (
    get_classification_models,
    train_and_tune_models,
    evaluate_kmeans_clustering,
    run_apriori_association_mining,
    RANDOM_STATE
)

print(f"Configured random state: {RANDOM_STATE}")


### 2. Algorithmic Suitability & Academic Commentary
Before training, we formalize why each chosen algorithm is theoretically suited to food adulteration risk modeling:

| Algorithm | Paradigm | Theoretical Suitability for Food Safety Risk Mining |
|---|---|---|
| **Decision Tree** | White-box Rule Induction | Generates transparent, human-readable Boolean splits. Field inspectors can translate nodes directly into legal statutory enforcement rules (e.g., if $pH > 7.5$ and $Moisture > 90\% \implies$ Contaminated). |
| **Random Forest** | Bagging Ensemble | Mitigates individual tree overfitting by aggregating de-correlated trees. Outstanding at modeling non-linear interactions between continuous variables and commodity classes. |
| **Naive Bayes** | Probabilistic (Bayesian) | Assumes class-conditional feature independence. Extremely computationally lightweight; computes real-time posterior probabilities $P(Adulterated \mid X)$ under Bayesian updating. |
| **Support Vector Machine (SVM)** | Maximum Margin Kernel | Projects feature vectors into high-dimensional space to identify the optimal separating hyperplane with maximum margin, resilient against sensory collinearity. |
| **Logistic Regression** | Generalized Linear Model | Benchmark probabilistic classifier producing calibrated log-odds ratios and direct sigmoid risk probabilities without distributional assumptions. |
| **K-Nearest Neighbors (KNN)** | Non-parametric Instance-Based | Classifies unlabelled test food items by querying geometric similarity against historical laboratory forensic samples stored in feature memory. |
| **Gradient Boosting** | Sequential Boosting Ensemble | Sequentially constructs weak learners that minimize residual pseudo-residuals, yielding high predictive precision on subtle, borderline adulteration cases. |


In [ ]:
# Load preprocessed training dataset
train_path = '../data/processed/train_preprocessed.csv'
df_train = pd.read_csv(train_path)

X_train = df_train.drop(columns=['is_adulterated']).values
y_train = df_train['is_adulterated'].values

print(f"Training feature matrix shape: {X_train.shape}")
print(f"Class distribution: {np.bincount(y_train)} (Balanced via training SMOTE)")


In [ ]:
# Execute 5-fold Stratified GridSearchCV for all 7 classifiers
cv_results = train_and_tune_models(X_train, y_train, cv=5, scoring='recall')

# Summarize best parameters and cross-validation recall
tuning_summary = []
for name, res in cv_results.items():
    tuning_summary.append({
        'Model Name': name,
        'Best CV Recall': round(float(res['best_cv_recall']), 4),
        'Optimal Hyperparameters': str(res['best_params'])
    })

df_tuning = pd.DataFrame(tuning_summary)
pd.set_option('display.max_colwidth', None)
df_tuning


---
### 3. Unsupervised K-Means Clustering Validation
We now test whether food samples naturally segregate into distinct clusters **without providing class labels** during training. We evaluate $k \in [2, 6]$ using Inertia (within-cluster sum of squares) and Silhouette Coefficients.


In [ ]:
# Run K-Means cluster analysis and save Figure 9
kmeans_out = evaluate_kmeans_clustering(
    X=X_train,
    y_true=y_train,
    k_range=range(2, 7),
    output_fig_path='../reports/figures/fig9_kmeans_elbow_silhouette.png'
)

display(Image(filename='../reports/figures/fig9_kmeans_elbow_silhouette.png'))


> **Analytical Interpretation (Figure 9 & K-Means Results):**  
> Figure 9 displays the dual validation curve for unsupervised K-Means clustering. 
> - **Elbow Curve (Inertia):** The inertia curve shows an inflection 'elbow' point precisely at $k=2$ (Inertia drops from 34,767.6 at $k=2$ to 27,389.3 at $k=3$), demonstrating that two clusters account for the predominant geometric variance.
> - **Silhouette Score:** The silhouette coefficient achieves its maximum value ($S = 0.4131$) at $k=2$, confirming optimal cluster separation and cohesion.
> - **Contingency Matrix ($k=2$):** Comparing unsupervised cluster assignments to ground-truth labels shows:
>   - Cluster 0: 3,168 Safe samples vs 7 Adulterated samples (99.78% purity).
>   - Cluster 1: 3,179 Adulterated samples vs 18 Safe samples (99.44% purity).
> This provides strong empirical confirmation that adulterated foods exhibit distinct multivariate signatures that naturally cluster apart from safe foods even without supervision.


---
### 4. Apriori Association Rule Mining
We apply the Apriori algorithm to discover frequent co-occurrence patterns among discretized operational indicators leading to adulteration.


In [ ]:
# Execute Apriori Association Mining on raw dataset
df_rules = run_apriori_association_mining(raw_csv_path='../data/raw/food_adulteration_raw.csv')
df_rules


### 5. Domain Interpretation of Discovered Association Rules
1. **Rule 1 (`NonPermitted_Preservative \implies Adulterated`):**  
   *Support = 15.92%, Confidence = 100.0%, Lift = 2.97.*  
   Any food sample containing unauthorized chemical preservatives (such as formalin in fish or salicylic acid in dairy) is universally classified as adulterated under Section 3(1)(a) of the FSS Act.
2. **Rule 2 (`NonPermitted_Preservative & Expired_Stock \implies Adulterated`):**  
   *Support = 9.32%, Confidence = 100.0%, Lift = 2.97.*  
   Compound violation indicating fraudulent attempts to extend the commercial shelf-life of expired batches using chemical masking agents.
3. **Rule 3 (`Expired_Stock & Moisture_Anomalous \implies Adulterated`):**  
   *Support = 10.70%, Confidence = 100.0%, Lift = 2.97.*  
   Captures physical degradation in expired stock (e.g. moisture absorption leading to microbial spoilage in grains or spices).
4. **Rule 4 (`Severe_Sensory_Defect & pH_Anomalous \implies Adulterated`):**  
   *Support = 11.85%, Confidence = 99.86%, Lift = 2.97.*  
   Confirms that simultaneous physical pH shifts and acute organoleptic failure are near-certain indicators of biological or chemical spoilage.

In **Notebook 05**, we evaluate all tuned models against the independent hold-out test set, construct ROC-AUC curves, compute confusion matrices, generate SHAP explanations, and select the final production model based on Recall.
